# 02: limpieza

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd
from src.config import RAW_FILE
data=pd.read_csv(RAW_FILE)

## Diccionario de datos

Dataset: **Telco Customer Churn** (7.043 clientes, 21 columnas). Cada fila es un cliente.

### Identificador

| Variable | Descripción | Tipo |
|---|---|---|
| `customerID` | Identificador único del cliente. **No se usa como variable del modelo** (solo sirve para identificar filas). | Texto |

### Datos demográficos

| Variable | Descripción | Valores | Tipo |
|---|---|---|---|
| `gender` | Género del cliente. | Male, Female | Categórica |
| `SeniorCitizen` | Si el cliente es una persona mayor (jubilado). | 1 = sí, 0 = no | Binaria (numérica) |
| `Partner` | Si el cliente tiene pareja. | Yes, No | Binaria |
| `Dependents` | Si el cliente tiene personas a su cargo (hijos, etc.). | Yes, No | Binaria |

### Antigüedad

| Variable | Descripción | Valores | Tipo |
|---|---|---|---|
| `tenure` | Número de meses que el cliente lleva en la compañía. | Entero (0 a 72) | Numérica |

### Servicios contratados

| Variable | Descripción | Valores | Tipo |
|---|---|---|---|
| `PhoneService` | Si tiene servicio de telefonía. | Yes, No | Binaria |
| `MultipleLines` | Si tiene varias líneas telefónicas. | Yes, No, No phone service | Categórica |
| `InternetService` | Tipo de servicio de internet contratado. | DSL, Fiber optic, No | Categórica |
| `OnlineSecurity` | Si tiene servicio de seguridad online. | Yes, No, No internet service | Categórica |
| `OnlineBackup` | Si tiene copia de seguridad online. | Yes, No, No internet service | Categórica |
| `DeviceProtection` | Si tiene protección de dispositivos. | Yes, No, No internet service | Categórica |
| `TechSupport` | Si tiene soporte técnico. | Yes, No, No internet service | Categórica |
| `StreamingTV` | Si tiene televisión en streaming. | Yes, No, No internet service | Categórica |
| `StreamingMovies` | Si tiene películas en streaming. | Yes, No, No internet service | Categórica |



### Contrato y facturación

| Variable | Descripción | Valores | Tipo |
|---|---|---|---|
| `Contract` | Duración del contrato. | Month-to-month, One year, Two year | Categórica ordinal |
| `PaperlessBilling` | Si recibe la factura en formato electrónico (sin papel). | Yes, No | Binaria |
| `PaymentMethod` | Método de pago. | Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic) | Categórica |
| `MonthlyCharges` | Importe cobrado al cliente cada mes. | Numérico (decimal) | Numérica |
| `TotalCharges` | Importe total cobrado al cliente durante toda su permanencia. | Numérico (decimal) | Numérica |


### Variable objetivo

| Variable | Descripción | Valores | Tipo |
|---|---|---|---|
| `Churn` | Si el cliente abandonó la compañía. **Es la variable a predecir.** | Yes, No (se codificará como 1 / 0) | Binaria |

In [19]:
#En primer lugar eliminamos la columna de ID
data=data.drop(columns=['customerID'])

Transformamos las variables binarias de Yes o No a valores númericos, en este caso son las siguientes: Churn, Partner, Dependants, PhoneService, PaperlessBilling.

Además, a la variable Gender le asociamos el número 1 en caso de ser hombre y el 0 en caso de ser mujer.

In [2]:
cols_bin=['Churn','Partner','Dependents','PhoneService','PaperlessBilling']
data[cols_bin]=data[cols_bin].apply(lambda s: s.map({'Yes': 1,'No': 0}))
data['gender']=data['gender'].map({'Male':1,'Female':0})

In [3]:
#veamos que hacemos con la columna MultipleLines
data.MultipleLines.value_counts()

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

In [4]:
#Definiremos No phone service como 0, No como 1 pues implica que solo tienen una línea, y MultipleLines lo definiremos como un 2.
data['MultipleLines']=data['MultipleLines'].map({"No":1,"Yes":2,"No phone service":0})


In [5]:
#Para las categorías que tienen Yes, No y No Internet Service lo que haremos es poner 1:Yes y 0 en cc
cols_internet = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection','TechSupport', 'StreamingTV', 'StreamingMovies']
data[cols_internet]=data[cols_internet].apply(lambda s: s.map({'Yes':1,'No':0,'No internet service':0}))

Ahora utilizaremos One Hot Encoding en la columna de Internet Service. Esto lo hacemos pues no existe una relación de orden entre DSL y Fiber. 

En este caso, utilizaremos la función get dummies de pandas. Pero en producción, seguramente utilicemos la función OneHotEncoding.

In [6]:
#Ahora utilizaremos One Hot Encoding en la columna de Internet Service. 

#En primer lugar, ordenamos la columna.esto lo hacemos para posteriormente poder hacer drop_first=True.
data['InternetService']=pd.Categorical(data['InternetService'],categories=['No','DSL','Fiber optic'])

data=pd.get_dummies(data,columns=['InternetService'],drop_first=True,dtype=int)

In [7]:
#Por ejemplo para Contract si vamos a utilizar 1,12,24 dependiendo de la duración. 
data['Contract']=data['Contract'].map({'Two year':24,'One year':12,'Month-to-month':1})

In [8]:
data['PaymentMethod']=pd.Categorical(data['PaymentMethod'])
data=pd.get_dummies(data,columns=['PaymentMethod'],dtype=int)

In [9]:
#Por último rellenaremos el campo TotalCharges que tiene 11 valores vacíos.
data['TotalCharges']=pd.to_numeric(data['TotalCharges'],errors='coerce').astype(float)

In [10]:
mediana=data['TotalCharges'].median()

In [11]:
#Existen valores atípicos por lo que imputaremos la mediana a estos 11 casos.
data['TotalCharges']=data['TotalCharges'].fillna(mediana)

In [12]:
#Por último, guardamos el DataFrame en la carpeta processed:
from src.config import PROCESSED_FILE
data.to_csv(PROCESSED_FILE,index=False)